In [53]:
import requests
import json
import time
from bs4 import BeautifulSoup
import newspaper
from newspaper import Config


In [54]:
def fetch_search_results(url, headers):
    """Fetch search results from the given URL using requests."""
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        return BeautifulSoup(response.content, 'html.parser')
    except requests.exceptions.RequestException as e:
        print(f"Error fetching search results: {e}")
        return None

def extract_results(soup, limit=10):
    """
    Extract the first `limit` search result links from the BeautifulSoup-parsed page.
    Note: You might need to adjust the selector if Bing's page structure changes.
    """
    return soup.find_all('a', class_='title', limit=limit) if soup else [] # how many articles to extract

def extract_article_with_newspaper(link, user_agent):
    """
    Use the newspaper library with a custom user agent to download, parse, and extract article content.
    Returns the full text and a summary (first 50 words).
    """
    try:
        # Create a configuration object and set your custom browser user agent.
        config = Config()
        config.browser_user_agent = user_agent
        article = newspaper.Article(link, config=config)
        article.download()
        article.parse()
        text = article.text
        summary = ' '.join(text.split()[:50])  # first 50 words as a summary
        
        # Attempt to get the publish date.
        publish_date = article.publish_date
        if publish_date:
            # Format the date as a string; adjust the format if needed.
            publish_date_str = publish_date.strftime("%Y-%m-%d %H:%M:%S")
        else:
            publish_date_str = "Unknown"
            
        return text, summary, publish_date_str
    except Exception as e:
        print(f"Error extracting article from {link}: {e}")
        return None, None, None

def fallback_extraction(link, headers):
    """
    Fallback extraction using BeautifulSoup.
    This method attempts to extract the page content by targeting common content containers.
    """
    try:
        response = requests.get(link, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        # Try to locate common container elements for article text.
        article_div = soup.find('div', class_='article-body') or soup.find('div', class_='content')
        if article_div:
            text = article_div.get_text(separator=' ', strip=True)
        else:
            # Fallback to extracting all text from the page.
            text = soup.get_text(separator=' ', strip=True)
        summary = ' '.join(text.split()[:50])
        publish_date_str = "Unknown"
        return text, summary, publish_date_str
    except Exception as e:
        print(f"Fallback extraction failed for {link}: {e}")
        return None, None, None 

def extract_article(link, user_agent, headers):
    """
    Attempts to extract article content using Newspaper.
    If extraction fails or the content is too short, fallback to BeautifulSoup extraction.
    """
    text, summary, publish_date = extract_article_with_newspaper(link, user_agent)
    # Use a threshold of 100 words; adjust this value based on your requirements.
    if not text or len(text.split()) < 100:
        print("Falling back to BeautifulSoup extraction due to insufficient content length.")
        text, summary, publish_date = fallback_extraction(link, headers)
    return text, summary, publish_date

def save_to_json(data, file_path):
    """Append the provided data as JSON into a file."""
    try:
        with open(file_path, 'a', encoding='utf-8') as f:
            json.dump(data, f, indent=4, ensure_ascii=False)
            f.write('\n')
    except IOError as e:
        print(f"Error saving to file: {e}")

def main():
    # Define a list of search terms.
    search_terms = ["Threat actors"]

    # Set your HTTP request headers; these help mimic a real browser.
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                      'AppleWebKit/537.36 (KHTML, like Gecko) '
                      'Chrome/91.0.4472.124 Safari/537.36'
    }
    user_agent = headers['User-Agent']
    
    # Define the output JSON file path.
    file_path = '/Users/jaytlinaskew/GitRepository/TimeSeries-Analysis/data/processed/BingSearchData.json'

    # Desired number of valid articles per search term.
    desired_valid_articles = 10

    for term in search_terms:
        print(f"\nSearching for: {term}")
        search_url = f"https://www.bing.com/news/search?q={term}"
        soup = fetch_search_results(search_url, headers)
        # Increase candidate limit to allow for some skipped links.
        candidate_links = extract_results(soup, limit=20)
        
        valid_articles_count = 0  # Reset count for each search term.
        for result in candidate_links:
            title = result.get_text(strip=True)
            link = result.get('href')
            print(f"Title: {title}")
            print(f"Link: {link}")
            timestamp = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

            # Attempt to extract the article with the combined method.
            content, summary, publish_date = extract_article(link, user_agent, headers)
            
            # Only count articles with sufficient content (at least 100 words).
            if content and len(content.split()) >= 100:
                print(f"Summary: {summary}")
                data = {
                    'search_term': term,
                    'timestamp': timestamp,
                    'title': title,
                    'link': link,
                    'content': content,
                    'summary': summary,
                    'publish_date': publish_date
                    }
                save_to_json(data, file_path)
                valid_articles_count += 1
                print(f"Valid articles so far: {valid_articles_count}/{desired_valid_articles}")
                # Stop processing candidate links once we've reached the desired count.
                if valid_articles_count >= desired_valid_articles:
                    break
            else:
                print("Excluding this link due to insufficient extracted content.")
                
        if valid_articles_count < desired_valid_articles:
            print(f"Only {valid_articles_count} valid articles were found for search term '{term}'.")

if __name__ == "__main__":
    main()



Searching for: Threat actors
Title: Threat Actors Setting Up Persistent Access to Hosts Hacked in CrushFTP Attacks
Link: https://www.securityweek.com/threat-actors-set-up-persistent-access-to-hosts-hacked-in-crushftp-attacks/
Error extracting article from https://www.securityweek.com/threat-actors-set-up-persistent-access-to-hosts-hacked-in-crushftp-attacks/: Article `download()` failed with 403 Client Error: Forbidden for url: https://www.securityweek.com/threat-actors-set-up-persistent-access-to-hosts-hacked-in-crushftp-attacks/ on URL https://www.securityweek.com/threat-actors-set-up-persistent-access-to-hosts-hacked-in-crushftp-attacks/
Falling back to BeautifulSoup extraction due to insufficient content length.
Fallback extraction failed for https://www.securityweek.com/threat-actors-set-up-persistent-access-to-hosts-hacked-in-crushftp-attacks/: 403 Client Error: Forbidden for url: https://www.securityweek.com/threat-actors-set-up-persistent-access-to-hosts-hacked-in-crushftp-att